# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This includes reading the Croissant schema and viewing essential dataset metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print key metadata
print(f"Dataset Title: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. The `mlcroissant` library exposes dataset structure through the Croissant schema, and referencing always uses the unique `@id` for each record set and field.

We'll first enumerate all Record Sets to get their `@id`, `name`, and `description`. For the main table, we'll also enumerate all fields and columns, showing their `@id`, `name`, and other details.

In [ ]:
# List all record sets in the dataset schema with their @id
print("Available Record Sets:")
for recordset in metadata.record_sets:
    print(f"- @id: {recordset.id}")
    print(f"  name: {getattr(recordset, 'name', 'N/A')}")
    print(f"  description: {getattr(recordset, 'description', '')}")

# For demonstration, pick the first RecordSet @id (main table)
main_recordset = metadata.record_sets[0]
main_recordset_id = main_recordset.id

print(f"\nFields and Columns for main RecordSet (@id: {main_recordset_id}):")
for field in main_recordset.fields:
    print(f"- field @id: {field.id}")
    print(f"  name: {getattr(field, 'name', 'N/A')}")
    print(f"  dataType: {getattr(field, 'data_type', 'N/A')}")
    if hasattr(field, 'columns'):
        print(f"  Columns:")
        for column in field.columns:
            print(f"    - column @id: {column.id}, name: {getattr(column, 'name', 'N/A')}")

## 3. Data Extraction
Load tabular data from each record set into a Pandas DataFrame for analysis. All referencing is done by the entities' `@id` fields.

Here, we extract each record set using its `@id`, collecting all records as a list (rows) and loading them into DataFrames.

In [ ]:
# Prepare list of record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print('Found Record Set @ids:', record_set_ids)

dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from Record Set '@id': {rsid}")

# Display columns for the main record set DataFrame and the first few rows
print(f"\nColumns in main record set (@id: {main_recordset_id}):\n", dataframes[main_recordset_id].columns.tolist())
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing steps. We'll reference all fields by their `@id` and adapt these examples using actual numeric fields and clinical groupings as available.

*Example: We'll filter by a numeric column (such as age or interval months), normalize it, and then group by an anatomical or biomarker column.*

In [ ]:
# Choose field @id for a numeric column (example: age, interval_between_diagnoses_months, etc.)
# Adjust these to the actual @id from your overview above
# Let's assume the main table has these @id values:
#   - Numeric field @id: 'https://api.app.sen.science/frontiers/7862866/col-interval_months'
#   - Group field @id:   'https://api.app.sen.science/frontiers/7862866/col-anatomical_site'

main_df = dataframes[main_recordset_id]
interval_months_id = 'https://api.app.sen.science/frontiers/7862866/col-interval_months'  # Replace with actual @id in your dataset
group_field_id = 'https://api.app.sen.science/frontiers/7862866/col-anatomical_site'      # Replace with actual @id in your dataset

# Check if these @ids exist in the columns
if interval_months_id in main_df.columns:
    numeric_field = interval_months_id
    # Filter where interval_months > 10
    threshold = 10
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field for the filtered records
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by anatomical site (if group field present)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field_id}:")
        display(grouped_df.head())
else:
    print(f"Numeric field '@id' {interval_months_id} not present in DataFrame columns. Use your data overview to select correct @ids.")

## 5. Visualization
Visualize the distribution of the selected numeric variable and its grouping by an anatomical or biomarker variable. All axes and labels reference columns by their Croissant `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if interval_months_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[interval_months_id], bins=20, kde=True)
    plt.title(f"Distribution of '{interval_months_id}'")
    plt.xlabel(interval_months_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by anatomical site
    if group_field_id in main_df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=interval_months_id, data=main_df)
        plt.title(f"'{interval_months_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(interval_months_id)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we have demonstrated loading the FAIR^2 dataset via its Croissant schema using `mlcroissant`, listing all record sets and fields by `@id`, extracting tabular data, and performing basic exploratory analysis and visualizations on clinical variables. 

**Key findings:**
- The dataset structure is clearly navigable by `@id` for each record set and field.
- Basic statistics and data processing (filtering, normalization, grouping) can be performed seamlessly with Pandas after loading.
- Visualization of numeric clinical intervals grouped by anatomical sites is possible for initial insights regarding clinical distributions.

For further statistical analysis or machine learning, continue using the entity `@id` for all field references and document all data processing steps referencing schema definitions for reproducibility.